In [13]:
# import the necessary libraries
import ollama
import pandas as pd
import os
import time
from tqdm import tqdm

## Setting up the file path for financial sarcasm data and the newly labelled data for experiment 1

In [ ]:

finance_data = "/Users/mariyajose/Documents/Sarcasm/DeepSeek/rq1/sarcasm_test_data.csv"
exp1_labeled = "/Users/mariyajose/Documents/Sarcasm/DeepSeek/rq1/exp1_labeled.csv"



In [15]:
# read the finance data into pandas dataframe fin_data
fin_data = pd.read_csv(finance_data)
print(f"Total rows before removing nulls: {len(fin_data)}")

Total rows before removing nulls: 590


## Preschesks before starting the process
#### 1. Check if there any file with the same name.If it exists check how many rows are already labelled and labbel the remaining ones
#### 2. if the file does ot exists start labelling the data from the beginning 


In [16]:
if os.path.exists(exp1_labeled):
    print("Found an existing file with the same name, continuing labeling...")
    new_labeled_df = pd.read_csv(exp1_labeled)
    
    if 'exp1_label' not in new_labeled_df.columns:
        new_labeled_df['exp1_label'] = pd.NA
else:
    print("File not found with the given name, creating a new file and starting labeling from the beginning")
    new_labeled_df = fin_data.copy()
    new_labeled_df['exp1_label'] = pd.NA

# Fixed index finder to handle pd.NA safely
def index_finder(df):
    for index, value in df['exp1_label'].items():
        if pd.isna(value):
            return index
        if value not in [0, 1, 0.0, 1.0]:
            return index
    return len(df)

start_index = index_finder(new_labeled_df)

if start_index >= len(new_labeled_df):
    print("\n The dataset is already fully labeled. No further labeling is required.")
else:
    print(f"\n Continuing labelling from row index: {start_index} out of {len(new_labeled_df)}")

File not found with the given name, creating a new file and starting labeling from the beginning

 Continuing labelling from row index: 0 out of 590


## Create a function that can pass the comment and parent comment through deepseek-r1:8b to label the data based on the given prompt, the temparature is kept as 0 to keep the answers stable even if the same experiment ran multiple times.Using this the model will choose the heighst probability label.

In [17]:
def labeller(parent_comment, comment):
    prompt = f"""
    Find out if the following Reply comment is a sarcastic response to the  parent comment?
    - The Parent comment: "{parent_comment}"
    - The Reply comment: "{comment}"

    Output ONLY the single digit 1 if it is sarcastic, or 0 if it is not sarcastic. Do not include any explanations or extra words.
    """
    try:
        result = ollama.chat(
            model='deepseek-r1:8b', 
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.0}
        )
        raw_result = result['message']['content'].strip()
        
        if "</think>" in raw_result:
            raw_result = raw_result.split("</think>")[-1].strip()
            
        if "1" in raw_result:
            return 1
        elif "0" in raw_result:
            return 0
        else:
            return 1 if "sarcastic" in raw_result.lower() else 0
            
    except Exception as e:
        raise RuntimeError(f"Ollama Call Error: {e}")



## Once the function is ready, after the checks iterate throgh all the remaining rows inside the dataframe if tsome of the data was already labelled, else start labeling from the first row.

In [ ]:

if start_index < len(new_labeled_df):
    print("\nIterating through remaining unlabelled rows via local Ollama instance...")
    start_time = time.time()

    non_labelled_rows = new_labeled_df.iloc[start_index:]

    for index, row in tqdm(non_labelled_rows.iterrows(), total=len(non_labelled_rows), desc="Processing Rows"):
        try:
            prediction = labeller(row['parent_comment'], row['comment'])
            new_labeled_df.at[index, 'exp1_label'] = prediction

            # Print the live update after processing the row
            print(f" Row {index} is successfully processed and labeled.")
            
        except Exception as e:
            print(f"\n Row {index} is skipped, failed with error: {e}. Moving to next row.")
            continue
        
        # After every 10 rows, save the dataframe to the CSV file to ensure progress is not lost
        if index % 10 == 0:
            new_labeled_df.to_csv(exp1_labeled, index=False)

    # Final save
    new_labeled_df.to_csv(exp1_labeled, index=False)

    end_time = time.time()
    total_seconds = end_time - start_time
    hours, minutes, seconds = int(total_seconds // 3600), int((total_seconds % 3600) // 60), int(total_seconds % 60)

    print("\n" + "="*50)
    print(f"Completed labelling the dataset. Total rows processed: {len(non_labelled_rows)}")
    print(f"Results are  saved to: {os.path.abspath(exp1_labeled)}")
    print(f"Total time taken to complete the process is : {hours}h {minutes}m {seconds}s")
    print("="*50)
else:
    print("\n The dataset is already fully labeled. No further labeling is required.")


Iterating through remaining unlabelled rows via local Ollama instance...


Processing Rows:   0%|          | 1/590 [00:22<3:40:24, 22.45s/it]

 Row 0 is successfully processed and labeled.


Processing Rows:   0%|          | 2/590 [00:36<2:48:42, 17.22s/it]

 Row 1 is successfully processed and labeled.


...

 Row 587 is successfully processed and labeled.


Processing Rows: 100%|█████████▉| 589/590 [3:23:36<00:25, 25.19s/it]

 Row 588 is successfully processed and labeled.


Processing Rows: 100%|██████████| 590/590 [3:23:55<00:00, 20.74s/it]

 Row 589 is successfully processed and labeled.

Completed labelling the dataset. Total rows processed: 590
Results are  saved to: /Users/mariyajose/Documents/Sarcasm/DeepSeek/rq1/exp2_labeled.csv
Total time taken to complete the process is : 3h 23m 55s


In [20]:
print("\n" + "="*50)
print(f"Completed labelling the dataset. Total rows processed: {len(non_labelled_rows)}")
print(f"Results are  saved to: {os.path.abspath(exp1_labeled)}")
print(f"Total time taken to complete the process is : {hours}h {minutes}m {seconds}s")
print("="*50)


Completed labelling the dataset. Total rows processed: 590
Results are  saved to: /Users/mariyajose/Documents/Sarcasm/DeepSeek/rq1/exp2_labeled.csv
Total time taken to complete the process is : 3h 23m 55s
